# 6. Deployment: Implementasi dan Diseminasi Sistem Geospasial Tutupan Lahan Sentinel-2 Jawa Timur

**Mata Kuliah**: Proyek Sains Data | **Fase CRISP-DM**: Phase 6 - Deployment  
**Sensor Citra**: Satelit Multi-Spektral Copernicus Sentinel-2 Level-2A (Reflektansi Permukaan / BOA)  
**Wilayah Studi**: Jawa Timur (Scene Surabaya, Sidoarjo, Gresik, dsk.)  
**Model Utama**: LightGBM Pixel Classifier (`model_lightgbm_piksel.pkl`)  
**Dimensi Fitur**: 13 Fitur Spektral Pasti (7 Band BOA + 6 Indeks Biofisik Teruji)  
**Aplikasi Produksi**: Streamlit Interactive GIS Dashboard ([GitHub Repo](https://github.com/IlhamInsight/sentinel2-jatim-lulc-app))  

---

## 6.1 Tujuan dan Peran Tahap Deployment dalam CRISP-DM

Tahap **Deployment** merupakan fase pamungkas dalam metodologi **CRISP-DM** (*Cross-Industry Standard Process for Data Mining*). Pada tahap ini, model prediktif terbaik yang telah dilatih pada Tahap 4 (Modeling) dan divalidasi secara ilmiah pada Tahap 5 (Evaluation) ditransformasikan menjadi **produk operasional yang siap pakai (*production-ready system*)**.

### Prinsip Pemisahan Dokumentasi & Aplikasi Interaktif:
1. **Jupyter Notebook (`6-deployment.ipynb`)**:
   - Berfungsi sebagai **dokumentasi teknis dan saintifik** tahap deployment pada buku laporan Jupyter Book / CRISP-DM.
   - Mendokumentasikan verifikasi model, arsitektur modul bersama (`shared/`), proses inferensi *full-raster* dari citra mentah Sentinel-2, pemotongan wilayah spasial (AOI windowing), penanganan ambang batas keyakinan (*confidence threshold*), dan validasi output GeoTIFF.
2. **Aplikasi Streamlit Interaktif (`app.py`)**:
   - Berfungsi sebagai **antarmuka pengguna interaktif (*interactive operational deployment*)** yang dapat diakses oleh praktisi GIS, akademisi, dan pembuat kebijakan.
   - Memungkinkan eksplorasi interaktif peta Leaflet, pengaturan ambang batas keyakinan, pemilihan wilayah spasial, serta inspeksi profil spektral per piksel secara *real-time*.
   - **Repositori Produksi Mandiri**: Sesuai standar arsitektur perangkat lunak, aplikasi Streamlit production dipisahkan ke repositori tersendiri di [https://github.com/IlhamInsight/sentinel2-jatim-lulc-app](https://github.com/IlhamInsight/sentinel2-jatim-lulc-app).


In [1]:
import os
import sys
import time
import json
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import from_bounds
from rasterio.warp import transform_bounds
import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
import seaborn as sns

BASE_DIR = os.path.abspath('')
UTS_DIR = os.path.dirname(BASE_DIR) if os.path.basename(BASE_DIR) == '6-deployment' else os.path.join(BASE_DIR, 'src', 'Tugas_UTS')

BAND_NAMES = ["B02", "B03", "B04", "B08", "B8A", "B11", "B12"]
INDEX_NAMES = ["NDVI", "MNDWI", "NDBI", "Mangrove_Index_8A_11_4", "EVI", "BSI"]
FEATURE_COLS = BAND_NAMES + INDEX_NAMES

CLASS_CONFIG = {
    0: {"key": "built", "name": "Bangunan", "alias": "Built-up", "color": "#ef4444", "rgb": (239, 68, 68), "desc": "Permukiman, industri, jalan, dan perkerasan beton."},
    1: {"key": "farm", "name": "Lahan Pertanian", "alias": "Farmland", "color": "#eab308", "rgb": (234, 179, 8), "desc": "Sawah irigasi, tegalan, ladang semusim, dan hortikultura."},
    2: {"key": "water", "name": "Perairan", "alias": "Water Body", "color": "#0284c7", "rgb": (2, 132, 199), "desc": "Selat Madura, laut pesisir, muara sungai, dan waduk."},
    3: {"key": "forest", "name": "Hutan Daratan", "alias": "Terrestrial Forest", "color": "#22c55e", "rgb": (34, 197, 94), "desc": "Hutan alami dan tutupan kanopi lebat dataran tinggi."},
    4: {"key": "mangrove", "name": "Hutan Mangrove", "alias": "Mangrove Forest", "color": "#06b6d4", "rgb": (6, 182, 212), "desc": "Vegetasi bakau pasang-surut pantai dan muara berlumpur."},
    255: {"key": "nodata", "name": "Unknown / Low Confidence", "alias": "Unknown/NoData", "color": "#000000", "rgb": (0, 0, 0), "desc": "Piksel NoData, awan/saturasi, atau di bawah ambang keyakinan."}
}
VALID_CLASSES = [0, 1, 2, 3, 4]

REGIONS_AOI = {
    "Seluruh Citra Scene (Surabaya - Sidoarjo - Gresik)": {
        "bounds": (112.50, -7.60, 113.00, -7.00), "center": (-7.30, 112.75), "zoom": 10,
        "title": "Scene Sentinel-2 Jawa Timur (Penuh)",
        "desc": "Wilayah cakupan penuh citra Sentinel-2 mencakup Surabaya, Sidoarjo, dan pesisir Gresik."
    },
    "Surabaya (Metropolitan & Wonorejo)": {
        "bounds": (112.60, -7.36, 112.85, -7.18), "center": (-7.26, 112.75), "zoom": 12,
        "title": "Kota Surabaya (Urban & Pesisir Timur)",
        "desc": "Pusat bisnis perkotaan, kawasan industri, pelabuhan, dan mangrove pesisir Wonorejo."
    },
    "Malang & Batu (Dataran Tinggi & Pegunungan)": {
        "bounds": (112.45, -8.05, 112.75, -7.80), "center": (-7.95, 112.63), "zoom": 11,
        "title": "Kawasan Dataran Tinggi Malang & Kota Batu",
        "desc": "Pegunungan Arjuno-Semeru dan hortikultura (di luar cakupan scene citra ini)."
    }
}

def compute_spectral_indices(b02, b03, b04, b08, b8a, b11, b12, eps=1e-6):
    b02, b03, b04 = np.asarray(b02, dtype=np.float32), np.asarray(b03, dtype=np.float32), np.asarray(b04, dtype=np.float32)
    b08, b8a = np.asarray(b08, dtype=np.float32), np.asarray(b8a, dtype=np.float32)
    b11, b12 = np.asarray(b11, dtype=np.float32), np.asarray(b12, dtype=np.float32)
    ndvi = np.clip((b08 - b04) / (b08 + b04 + eps), -1.0, 1.0)
    mndwi = np.clip((b03 - b11) / (b03 + b11 + eps), -1.0, 1.0)
    ndbi = np.clip((b11 - b08) / (b11 + b08 + eps), -1.0, 1.0)
    mangrove = np.clip((b8a - b04) / (b11 + eps), -10.0, 10.0)
    evi_denom = b08 + 6.0 * b04 - 7.5 * b02 + 1.0
    safe_denom = np.where(np.abs(evi_denom) < 1e-4, 1e-4, evi_denom)
    evi = np.clip(2.5 * (b08 - b04) / safe_denom, -3.0, 3.0)
    bsi = np.clip(((b11 + b04) - (b08 + b02)) / ((b11 + b04) + (b08 + b02) + eps), -1.0, 1.0)
    return {"NDVI": ndvi, "MNDWI": mndwi, "NDBI": ndbi, "Mangrove_Index_8A_11_4": mangrove, "EVI": evi, "BSI": bsi}

def extract_13_features(bands_array, eps=1e-6):
    arr = np.asarray(bands_array, dtype=np.float32)
    if arr.ndim == 3 and arr.shape[0] == 7:
        _, h, w = arr.shape
        idx = compute_spectral_indices(arr[0], arr[1], arr[2], arr[3], arr[4], arr[5], arr[6], eps=eps)
        feature_stack = np.stack([
            arr[0], arr[1], arr[2], arr[3], arr[4], arr[5], arr[6],
            idx["NDVI"], idx["MNDWI"], idx["NDBI"], idx["Mangrove_Index_8A_11_4"], idx["EVI"], idx["BSI"]
        ], axis=-1)
        return feature_stack.reshape(-1, 13), (h, w)
    raise ValueError(f"Bentuk bands_array tidak valid: {arr.shape}")

def validate_model(mdl):
    if not hasattr(mdl, "classes_") or list(mdl.classes_) != VALID_CLASSES:
        raise ValueError(f"Kelas model tidak sesuai! Ekspektasi {VALID_CLASSES}")
    if not hasattr(mdl, "n_features_in_") or mdl.n_features_in_ != len(FEATURE_COLS):
        raise ValueError(f"Jumlah fitur tidak sesuai! Ekspektasi {len(FEATURE_COLS)}")
    if not hasattr(mdl, "predict_proba"):
        raise ValueError("Model tidak mendukung fungsi predict_proba!")
    return True

def check_raster_coverage(raster_bounds, aoi_bounds):
    r_left, r_bottom, r_right, r_top = raster_bounds
    aoi_left, aoi_bottom, aoi_right, aoi_top = aoi_bounds
    if aoi_left >= r_right or aoi_right <= r_left or aoi_bottom >= r_top or aoi_top <= r_bottom:
        return False, None, "Wilayah tidak tercakup oleh citra Sentinel-2 yang tersedia."
    inter = (max(r_left, aoi_left), max(r_bottom, aoi_bottom), min(r_right, aoi_right), min(r_top, aoi_top))
    if inter[0] >= inter[2] or inter[1] >= inter[3]:
        return False, None, "Wilayah tidak tercakup oleh citra Sentinel-2 yang tersedia."
    return True, inter, "Wilayah berhasil tercakup oleh citra Sentinel-2."

def read_sentinel2_raster(tif_path, aoi_bounds=None):
    if not os.path.exists(tif_path):
        raise FileNotFoundError(f"File GeoTIFF tidak ditemukan: {tif_path}")
    with rasterio.open(tif_path) as src:
        crs, bounds, prof = src.crs, src.bounds, src.profile.copy()
        if crs and crs.to_epsg() != 4326:
            r_box = transform_bounds(crs, "EPSG:4326", bounds.left, bounds.bottom, bounds.right, bounds.top)
        else:
            r_box = (bounds.left, bounds.bottom, bounds.right, bounds.top)
        if aoi_bounds is not None:
            ok, inter, _ = check_raster_coverage(r_box, aoi_bounds)
            if not ok:
                raise ValueError("Wilayah tidak tercakup oleh citra Sentinel-2 yang tersedia.")
            win = from_bounds(*inter, transform=src.transform).round_offsets().round_lengths()
            bands = src.read(window=win)
            prof.update({"height": bands.shape[1], "width": bands.shape[2], "transform": rasterio.windows.transform(win, src.transform)})
            return bands.astype(np.float32), prof, rasterio.windows.bounds(win, src.transform), src.transform
        return src.read().astype(np.float32), prof, r_box, src.transform

def run_full_raster_inference(mdl, bands, confidence_threshold=0.65):
    validate_model(mdl)
    t0 = time.time()
    _, h, w = bands.shape
    X, _ = extract_13_features(bands)
    invalid_mask = np.isnan(X).any(axis=1) | np.isinf(X).any(axis=1) | (X[:, :7] <= 0.0).all(axis=1)
    X_clean = np.nan_to_num(X, nan=0.0, posinf=1.0, neginf=0.0)
    probs = mdl.predict_proba(X_clean)
    max_idx = np.argmax(probs, axis=1)
    max_prob = np.max(probs, axis=1)
    preds = mdl.classes_[max_idx].copy()
    preds[max_prob < confidence_threshold] = 255
    preds[invalid_mask] = 255
    max_prob[invalid_mask] = 0.0
    cls_raster = preds.reshape(h, w).astype(np.uint8)
    conf_raster = max_prob.reshape(h, w).astype(np.float32)
    
    unique, counts = np.unique(cls_raster, return_counts=True)
    count_dict = dict(zip(unique.tolist(), counts.tolist()))
    class_stats = {}
    total_pixels = h * w
    for c in [0, 1, 2, 3, 4, 255]:
        cnt = count_dict.get(c, 0)
        class_stats[c] = {
            "name": CLASS_CONFIG[c]["name"],
            "alias": CLASS_CONFIG[c]["alias"],
            "color": CLASS_CONFIG[c]["color"],
            "count": cnt,
            "percent": (cnt / total_pixels) * 100.0
        }
        
    return {
        "classified_raster": cls_raster,
        "confidence_raster": conf_raster,
        "class_stats": class_stats,
        "inference_time_sec": time.time() - t0,
        "total_pixels": total_pixels,
        "shape": (h, w)
    }

def save_inference_geotiffs(classified_raster, confidence_raster, base_profile, output_dir, prefix="classified"):
    os.makedirs(output_dir, exist_ok=True)
    h, w = classified_raster.shape
    c_prof = base_profile.copy()
    c_prof.update({"driver": "GTiff", "height": h, "width": w, "count": 1, "dtype": rasterio.uint8, "nodata": 255, "compress": "lzw"})
    c_path = os.path.join(output_dir, f"{prefix}_lulc.tif")
    with rasterio.open(c_path, "w", **c_prof) as dst:
        dst.write(classified_raster.astype(np.uint8), 1)

    cf_prof = base_profile.copy()
    cf_prof.update({"driver": "GTiff", "height": h, "width": w, "count": 1, "dtype": rasterio.float32, "nodata": 0.0, "compress": "lzw"})
    cf_path = os.path.join(output_dir, "confidence.tif")
    with rasterio.open(cf_path, "w", **cf_prof) as dst:
        dst.write(confidence_raster.astype(np.float32), 1)
    return c_path, cf_path

print('[OK] Modul inferensi geospasial tahap Deployment berhasil diinisialisasi.')



[OK] Modul geospasial dan dependensi tahap Deployment berhasil dimuat.


## 6.2 Pemuatan dan Validasi Integritas Model Final LightGBM

Sesuai metodologi CRISP-DM, model utama yang digunakan untuk inferensi raster geospasial resolusi tinggi 10 meter adalah **LightGBM Pixel Classifier** (`model_lightgbm_piksel.pkl`).

Sebelum model digunakan untuk inferensi seluruh piksel raster, dilakukan validasi ketat terhadap:
1. `model.classes_`: Memastikan model memiliki tepat 5 kelas target `[0, 1, 2, 3, 4]`.
2. `model.n_features_in_`: Memastikan model menerima tepat 13 fitur masukan tanpa perubahan skema.


In [2]:
model_dir = os.path.join(UTS_DIR, '4-modeling', 'models')
model_path = os.path.join(model_dir, 'model_lightgbm_piksel.pkl')
meta_path = os.path.join(model_dir, 'metadata_lightgbm_piksel.json')

# Pemuatan artefak model
model = joblib.load(model_path)
validate_model(model)

with open(meta_path, 'r', encoding='utf-8') as f:
    model_metadata = json.load(f)

print('=' * 75)
print('STATUS VALIDASI INTEGRITAS MODEL FINAL (LIGHTGBM PIKSEL 10M):')
print(f'• File Model             : {os.path.basename(model_path)} ({os.path.getsize(model_path)/1024:.1f} KB)')
print(f'• Nama Model             : {model_metadata.get("nama_model")}')
print(f'• Kelas Model (classes_) : {list(model.classes_)} (Tepat 5 Kelas Target)')
print(f'• Jumlah Fitur Masukan   : {model.n_features_in_} Fitur Spektral Pasti')
print(f'• Daftar 13 Fitur Pasti  : {FEATURE_COLS}')
print('=' * 75)


STATUS VALIDASI INTEGRITAS MODEL FINAL (LIGHTGBM PIKSEL 10M):
• File Model             : model_lightgbm_piksel.pkl (3325.2 KB)
• Nama Model             : LightGBM Pixel Classifier (Tuned Single Model - 10m Resolution)
• Kelas Model (classes_) : [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)] (Tepat 5 Kelas Target)
• Jumlah Fitur Masukan   : 13 Fitur Spektral Pasti
• Daftar 13 Fitur Pasti  : ['B02', 'B03', 'B04', 'B08', 'B8A', 'B11', 'B12', 'NDVI', 'MNDWI', 'NDBI', 'Mangrove_Index_8A_11_4', 'EVI', 'BSI']


D:\university\semester 5_book\proyek sain data\venv\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.9.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


## 6.3 Pemuatan Citra Geospasial Asli Sentinel-2 Menggunakan Rasterio

Sumber data inferensi adalah citra raster multi-spektral asli **Sentinel-2 Level-2A** (`sentinel2_jatim_scene.tif`), bukan raster klasifikasi lama.
Citra dibaca menggunakan pustaka standar geospasial **`rasterio`** guna mempertahankan metadata proyeksi, koordinat, transform affine, dan resolusi spasial asli.


In [3]:
raw_raster_path = os.path.join(UTS_DIR, '2-data_understanding', 'extracted_data', 'sentinel2_jatim_scene.tif')

with rasterio.open(raw_raster_path) as src:
    raster_crs = src.crs
    raster_bounds = src.bounds
    raster_transform = src.transform
    raster_shape = (src.count, src.height, src.width)
    raster_res = src.res

print('=' * 75)
print('METADATA RASTER CITRA SENTINEL-2 ASLI (sentinel2_jatim_scene.tif):')
print(f'• Driver Format          : GeoTIFF')
print(f'• Dimensi Piksel         : {raster_shape[2]} x {raster_shape[1]} piksel ({raster_shape[2]*raster_shape[1]:,} piksel total)')
print(f'• Jumlah Band Spektral   : {raster_shape[0]} Band (B02, B03, B04, B08, B8A, B11, B12)')
print(f'• Sistem Koordinat (CRS) : {raster_crs}')
print(f'• Bounding Box Geografis : {raster_bounds}')
print(f'• Resolusi Spasial Piksel: {raster_res[0]:.6f}° x {raster_res[1]:.6f}° (~10 meter)')
print('=' * 75)

# Visualisasi Komposit Warna Citra Asli (True Color vs False Color CIR)
bands_all, prof_all, bounds_all, trans_all = read_sentinel2_raster(raw_raster_path)

def norm_band(b):
    p2, p98 = np.nanpercentile(b, (2, 98))
    return np.clip((b - p2) / (p98 - p2 + 1e-6), 0, 1)

rgb_true = np.dstack([norm_band(bands_all[2]), norm_band(bands_all[1]), norm_band(bands_all[0])])
rgb_cir = np.dstack([norm_band(bands_all[3]), norm_band(bands_all[2]), norm_band(bands_all[1])])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
ax1.imshow(rgb_true)
ax1.set_title('Komposit True Color (B04-Red, B03-Green, B02-Blue)', fontsize=11, fontweight="bold")
ax1.axis('off')

ax2.imshow(rgb_cir)
ax2.set_title('Komposit False Color CIR (B08-NIR, B04-Red, B03-Green)', fontsize=11, fontweight="bold")
ax2.axis('off')
plt.tight_layout()
plt.show()
plt.close()


METADATA RASTER CITRA SENTINEL-2 ASLI (sentinel2_jatim_scene.tif):
• Driver Format          : GeoTIFF
• Dimensi Piksel         : 1024 x 1024 piksel (1,048,576 piksel total)
• Jumlah Band Spektral   : 7 Band (B02, B03, B04, B08, B8A, B11, B12)
• Sistem Koordinat (CRS) : EPSG:4326
• Bounding Box Geografis : BoundingBox(left=112.5, bottom=-7.6, right=113.0, top=-7.0)
• Resolusi Spasial Piksel: 0.000488° x 0.000586° (~10 meter)


C:\Users\user\AppData\Local\Temp\ipykernel_10968\1783392191.py:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6.4 Pipeline Inferensi Full-Raster dan Penanganan Ambang Batas Keyakinan

Tahap inferensi mengeksekusi urutan pipeline ilmiah terpadu:
$$\text{GeoTIFF (7 Band)} \xrightarrow{\text{6 Indeks}} \text{13 Fitur Spektral} \xrightarrow{\text{Reshape } N \times 13} \text{LightGBM.predict\_proba()} \xrightarrow{\text{Confidence Filtering}} \text{Reshape } H \times W$$

### Penanganan Ambang Batas Keyakinan (Confidence Threshold):
- Probabilitas maksimum tiap piksel dievaluasi: $\max_c P(y = c \mid \mathbf{x})$.
- Jika $\max_c P < \tau$ (di mana $\tau = 0.40$), piksel ditetapkan sebagai **Kelas 255** (*Unknown / Low Confidence / NoData*).
- Kode numerik kelas tutupan lahan:
  - `0`: Bangunan (*Built-up*)
  - `1`: Lahan Pertanian (*Farmland*)
  - `2`: Perairan (*Water Body*)
  - `3`: Hutan Daratan (*Terrestrial Forest*)
  - `4`: Hutan Mangrove (*Mangrove Forest*)
  - `255`: Unknown / Low Confidence (bukan kelas LULC keenam).


In [4]:
# Eksekusi inferensi penuh pada 1,048,576 piksel
t_start = time.time()
infer_result = run_full_raster_inference(model, bands_all, confidence_threshold=0.40)
t_elapsed = time.time() - t_start

classified_raster = infer_result['classified_raster']
confidence_raster = infer_result['confidence_raster']
stats = infer_result['class_stats']

print(f'[SUKSES] Inferensi Full-Raster selesai dalam {t_elapsed:.2f} detik.')
print(f'• Total Piksel Diproses : {infer_result["total_pixels"]:,} piksel')
print(f'• Resolusi Grid Keluaran: {classified_raster.shape[0]} baris x {classified_raster.shape[1]} kolom')
print(f'• Nilai Unik Piksel     : {np.unique(classified_raster).tolist()} (0..4, 255)')


[SUKSES] Inferensi Full-Raster selesai dalam 11.96 detik.
• Total Piksel Diproses : 1,048,576 piksel
• Resolusi Grid Keluaran: 1024 baris x 1024 kolom
• Nilai Unik Piksel     : [0, 1, 2, 3, 4, 255] (0..4, 255)


## 6.5 Analisis Distribusi Tutupan Lahan & Visualisasi Peta Spasial

Hasil prediksi model disajikan melalui visualisasi spasial peta tematik klasifikasi LULC dan peta distribusi keyakinan (*confidence map*), dilengkapi dengan tabel proporsi luas tutupan lahan.


In [5]:
# 1. Tabel Distribusi 5 Kelas LULC
dist_records = []
for c in [0, 1, 2, 3, 4, 255]:
    st_c = stats[c]
    dist_records.append({
        'Kode': c,
        'Nama Kelas': st_c['name'],
        'Alias Internasional': st_c['alias'],
        'Jumlah Piksel': f"{st_c['count']:,}",
        'Proporsi Luas (%)': f"{st_c['percent']:.2f}%"
    })

dist_df = pd.DataFrame(dist_records)
print('DISTRIBUSI PREDIKSI TUTUPAN LAHAN (FULL SCENE SENTINEL-2):')
display(dist_df)

# 2. Visualisasi Peta Klasifikasi & Peta Keyakinan
cmap_colors = ['#EF4444', '#D97706', '#0284C7', '#16A34A', '#0D9488', '#000000']
cmap = ListedColormap(cmap_colors)
bounds_bins = [-0.5, 0.5, 1.5, 2.5, 3.5, 4.5, 255.5]
norm = BoundaryNorm(bounds_bins, cmap.N)

fig, (ax_lulc, ax_conf) = plt.subplots(1, 2, figsize=(16, 7))

# Peta LULC
im_lulc = ax_lulc.imshow(classified_raster, cmap=cmap, norm=norm)
ax_lulc.set_title('Peta Klasifikasi Tutupan Lahan Sentinel-2 (LightGBM 10m)', fontsize=12, fontweight="bold", pad=10)
ax_lulc.axis('off')

# Legenda LULC
cbar = fig.colorbar(im_lulc, ax=ax_lulc, fraction=0.046, pad=0.04, ticks=[0, 1, 2, 3, 4, 255])
cbar.ax.set_yticklabels(['Bangunan', 'Pertanian', 'Perairan', 'Hutan', 'Mangrove', 'Low Conf / NoData'])

# Peta Keyakinan (Confidence)
im_conf = ax_conf.imshow(confidence_raster, cmap='viridis', vmin=0.4, vmax=1.0)
ax_conf.set_title('Peta Keyakinan Prediksi Model (Posterior Confidence Score)', fontsize=12, fontweight="bold", pad=10)
ax_conf.axis('off')
cbar_conf = fig.colorbar(im_conf, ax=ax_conf, fraction=0.046, pad=0.04)
cbar_conf.set_label('Confidence (0.40 - 1.00)', fontsize=10)

plt.tight_layout()
plt.show()
plt.close()


DISTRIBUSI PREDIKSI TUTUPAN LAHAN (FULL SCENE SENTINEL-2):


,Kode,Nama Kelas,Alias Internasional,Jumlah Piksel,Proporsi Luas (%)
0,0,Bangunan,Built-up,"301,008",28.71%
1,1,Lahan Pertanian,Farmland,"257,385",24.55%
2,2,Perairan,Water Body,"344,196",32.83%
3,3,Hutan Daratan,Terrestrial Forest,"101,405",9.67%
4,4,Hutan Mangrove,Mangrove Forest,"44,125",4.21%
5,255,Unknown / Low Confidence,Unknown/NoData,457,0.04%


C:\Users\user\AppData\Local\Temp\ipykernel_10968\2728133974.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6.6 Ekspor Hasil Inferensi Menjadi GeoTIFF Terkalibrasi Spasial

Hasil prediksi langsung diekspor ke file GeoTIFF dengan mempertahankan parameter georeferensi asli:
1. `classified_lulc.tif`: Raster kelas tutupan lahan (`uint8`, nodata=`255`).
2. `confidence.tif`: Raster tingkat keyakinan posterior (`float32`, nodata=`0.0`).


In [6]:
outputs_dir = os.path.join(UTS_DIR, '6-deployment', 'outputs')
class_path, conf_path = save_inference_geotiffs(
    classified_raster, confidence_raster, prof_all, outputs_dir, prefix='classified'
)

print('=' * 75)
print('VERIFIKASI OUTPUT RASTER GEOSPASIAL:')
print(f'1. Classified Raster : {class_path} ({os.path.getsize(class_path)/1024:.1f} KB)')
print(f'2. Confidence Raster : {conf_path} ({os.path.getsize(conf_path)/1024:.1f} KB)')

with rasterio.open(class_path) as chk:
    print(f'• Validasi CRS       : {chk.crs}')
    print(f'• Validasi Bounding  : {chk.bounds}')
    print(f'• Validasi Resolusi  : {chk.res}')
    print(f'• Validasi NoData    : {chk.nodata}')
    assert chk.nodata == 255, "NoData GeoTIFF harus 255!"
print('[OK] Seluruh file GeoTIFF terkalibrasi spasial dengan sempurna.')
print('=' * 75)


VERIFIKASI OUTPUT RASTER GEOSPASIAL:


1. Classified Raster : D:\university\semester 5_book\proyek sain data\src\Tugas_UTS\6-deployment\outputs\classified_lulc.tif (187.4 KB)
2. Confidence Raster : D:\university\semester 5_book\proyek sain data\src\Tugas_UTS\6-deployment\outputs\confidence.tif (2879.7 KB)
• Validasi CRS       : EPSG:4326
• Validasi Bounding  : BoundingBox(left=112.5, bottom=-7.6, right=113.0, top=-7.0)
• Validasi Resolusi  : (0.00048828125, 0.0005859374999999997)
• Validasi NoData    : 255.0
[OK] Seluruh file GeoTIFF terkalibrasi spasial dengan sempurna.


## 6.7 Pemotongan Wilayah (AOI Windowing) dan Pengecekan Cakupan Spasial

Untuk menguji generalisasi dan kepatuhan sistem pada batas wilayah:
1. **Wilayah Tercakup (Contoh: Surabaya)**: Sistem memotong jendela spasial (*window clipping*) dari GeoTIFF asli dan melakukan inferensi khusus pada koordinat tersebut.
2. **Wilayah di Luar Cakupan (Contoh: Malang / Batu)**: Sistem mendeteksi tidak adanya irisan koordinat dan menampilkan pesan:
   `"Wilayah tidak tercakup oleh citra Sentinel-2 yang tersedia."` tanpa melakukan klasifikasi palsu.


In [7]:
# 1. Wilayah Tercakup: Surabaya Metropolitan & Pesisir Timur
surabaya_aoi = REGIONS_AOI['Surabaya (Metropolitan & Wonorejo)']['bounds']
surabaya_bands, surabaya_prof, surabaya_bounds, surabaya_trans = read_sentinel2_raster(raw_raster_path, aoi_bounds=surabaya_aoi)

surabaya_infer = run_full_raster_inference(model, surabaya_bands, confidence_threshold=0.40)
print(f'• Sub-wilayah Surabaya : Diproses {surabaya_infer["total_pixels"]:,} piksel ({surabaya_infer["inference_time_sec"]:.2f} detik)')
print(f'  - Bangunan (Built-up) : {surabaya_infer["class_stats"][0]["percent"]:.2f}%')
print(f'  - Mangrove Pesisir    : {surabaya_infer["class_stats"][4]["percent"]:.2f}%')

# 2. Wilayah di Luar Cakupan: Kawasan Malang & Batu
malang_aoi = REGIONS_AOI['Malang & Batu (Dataran Tinggi & Pegunungan)']['bounds']
is_covered, inter_box, coverage_msg = check_raster_coverage(raster_bounds, malang_aoi)

print(f'\n• Pengecekan Wilayah Malang & Batu:')
print(f'  - Status Cakupan     : {is_covered}')
print(f'  - Respon Sistem      : "{coverage_msg}"')
if not is_covered:
    print('  [BENAR] Sistem menolak klasifikasi palsu pada area yang tidak tercakup citra.')


• Sub-wilayah Surabaya : Diproses 157,184 piksel (2.04 detik)
  - Bangunan (Built-up) : 48.22%
  - Mangrove Pesisir    : 3.13%

• Pengecekan Wilayah Malang & Batu:
  - Status Cakupan     : False
  - Respon Sistem      : "Wilayah tidak tercakup oleh citra Sentinel-2 yang tersedia."
  [BENAR] Sistem menolak klasifikasi palsu pada area yang tidak tercakup citra.


## 6.8 Dokumentasi Aplikasi Streamlit & Tautan Repositori Produksi

Aplikasi interaktif utama yang digunakan oleh pengguna akhir dikembangkan menggunakan **Streamlit** dan **Leaflet.js** pada file [`src/Tugas_UTS/6-deployment/app.py`](file:///d:/university/semester%205_book/proyek%20sain%20data/src/Tugas_UTS/6-deployment/app.py).

### Fitur Utama Aplikasi Streamlit:
1. **Full-Raster Trigger Button**: Inferensi hanya dijalankan ketika pengguna mengklik tombol *'Jalankan Inferensi Raster'*.
2. **Pengecekan Cakupan Wilayah Otomatis**: Menolak klasifikasi jika wilayah berada di luar batas citra.
3. **Peta Interaktif Leaflet**: Memproyeksikan layer raster hasil prediksi terbaru langsung di atas basemap satelit resolusi tinggi.
4. **Kontrol Layer Dinamis**: Checkbox kelas tutupan lahan untuk menyembunyikan/menampilkan kelas tanpa mengubah hasil prediksi model.
5. **Single-Pixel Profiler**: Memungkinkan pengguna memasukkan nilai reflektansi 7 band untuk melihat kurva spektral dan probabilitas posterior secara *real-time*.
6. **Unduh GeoTIFF**: Tombol unduh file `classified_lulc.tif` dan `confidence.tif`.

---

### Tautan Repositori Produksi Mandiri:
Aplikasi Streamlit production dipisahkan ke repositori independen yang siap dideploy ke Streamlit Cloud:
* **Tautan GitHub**: [https://github.com/IlhamInsight/sentinel2-jatim-lulc-app](https://github.com/IlhamInsight/sentinel2-jatim-lulc-app)


* **Perintah Menjalankan Aplikasi Lokal**:
  ```bash
  streamlit run src/Tugas_UTS/6-deployment/app.py
  ```

**URL DEPLOY:** https://sentinel2-jatim-lulc-app-53k6uykemnmygpqkcbyafm.streamlit.app/ 
